# B.O.S.S. — YOLOv8 Fine-tuning su Cityscapes per rilevamento ostacoli outdoor

**Pipeline:**
1. Setup ambiente e librerie
2. Download Cityscapes e conversione in formato YOLO
3. Definizione classi ostacoli e filtraggio dataset
4. Training YOLOv8n (o altro modello)
5. Valutazione su frame di test (`recordings/20260514_092949/frames`)
6. Metriche: mAP, F1, Precision, Recall, Confusion Matrix

---
**Nota:** Le celle sono progettate per essere autonome e modificabili. Per cambiare modello basta modificare `MODEL_PATH` nella cella di configurazione.

## Cella 0 — Setup ambiente (Colab / Kaggle / Locale)

**Esegui SOLO su Colab o Kaggle.** Salta su esecuzione locale.

**Colab:** monta Google Drive. Su Drive in `MyDrive/Lavoro/code/` servono:
`leftImg8bit_trainvaltest.zip`, `gtFine_trainvaltest.zip`, `BOSS_recordings.yolov8.zip`, `recordings.zip`, `yolov8n.pt`

**Kaggle:** aggiungi i seguenti dataset come input al notebook (slug esatti):
- `cityscapes_leftImg8bit_trainvaltest` — contiene `leftImg8bit/train/`, `leftImg8bit/val/`, `leftImg8bit/test/`
- `cityscapes_gtfine_trainvaltest` — contiene `gtFine/train/`, `gtFine/val/`, `gtFine/test/`
- `boss_recordings1` — contiene:
  - `BOSS_recordings/` (ground truth Roboflow: `train/images/`, `train/labels/`, `data.yaml`)
  - `recordings/20260514_092116/frames/` e `recordings/20260514_092949/frames/` (frame grezzi per inferenza)

In [16]:
import os, shutil
from pathlib import Path

IS_COLAB  = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lavoro/code")
    os.chdir(WORK_DIR)
    print("Colab. Working dir:", os.getcwd())
elif IS_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
    os.chdir(WORK_DIR)
    print("Kaggle. Working dir:", os.getcwd())
    print("Input disponibili:", list(Path("/kaggle/input").iterdir()))
else:
    print("Locale. Cella saltata.")


Kaggle. Working dir: /kaggle/working
Input disponibili: [PosixPath('/kaggle/input/datasets')]


## Cella 1 — Installazione dipendenze
Eseguire solo la prima volta o in un nuovo ambiente.

In [17]:
# Su Kaggle torch/torchvision/numpy/opencv sono preinstallati.
# Installa solo le librerie mancanti.
%pip install ultralytics pyyaml tqdm --quiet

# Su Colab o locale (torch non preinstallato), decommenta:
# %pip install torch torchvision --quiet
# %pip install ultralytics opencv-python Pillow matplotlib seaborn numpy pandas tqdm pyyaml --quiet


Note: you may need to restart the kernel to use updated packages.


## Cella 2 — Import librerie

In [18]:
import os                         # operazioni su file e directory
import shutil                     # copia/spostamento file
import json                       # parsing file JSON (annotazioni Cityscapes)
import glob                       # ricerca file con pattern
from pathlib import Path          # gestione percorsi cross-platform

import numpy as np                # operazioni su array e calcolo numerico
import pandas as pd               # strutture dati tabellari per metriche
import cv2                        # OpenCV: lettura/scrittura immagini, disegno
from PIL import Image             # Pillow: manipolazione immagini ad alto livello

import matplotlib.pyplot as plt   # grafici 2D (curve, bar chart)
import matplotlib.patches as mpatches  # elementi grafici personalizzati nei plot
import seaborn as sns             # heatmap e grafici statistici (confusion matrix)

from tqdm import tqdm             # barre di avanzamento per loop su dataset

from ultralytics import YOLO      # modello YOLOv8: training, val, predict

import warnings
warnings.filterwarnings('ignore')

## Cella 3 — Configurazione globale
**Modifica questa cella per cambiare modello, classi o percorsi.**

In [19]:
# ============================================================
# CONFIGURAZIONE — modifica questi valori per adattare la pipeline
# ============================================================

MODEL_PATH = "yolov8n.pt"

import torch

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    BASE_DIR  = Path("/kaggle/working")
else:
    INPUT_DIR = Path(".").resolve()
    BASE_DIR  = Path(".").resolve()

DATASET_DIR = BASE_DIR / "dataset_boss"  # dataset YOLO convertito da Cityscapes
GT_EXTRACTED = BASE_DIR / "ground_truth_boss"

if IS_KAGGLE:
    CITYSCAPES_LEFTIMG = "/kaggle/input/datasets/chrisviviers/cityscapes-leftimg8bit-trainvaltest/leftImg8bit"
    CITYSCAPES_GTFINE  = "/kaggle/input/datasets/lorenzoverdura/cityscapes-gtfine-trainvaltest/gtFine"
    GT_DIR         = "/kaggle/input/datasets/lorenzoverdura/boss-recordings1/BOSS_recordings.yolov8"
    TEST_FRAMES_DIR = "/kaggle/input/datasets/lorenzoverdura/boss-recordings1/recordings/recordings/20260514_092949/frames"
else:
    # Locale/Colab: Cityscapes estratto dagli ZIP in BASE_DIR
    CITYSCAPES_LEFTIMG = BASE_DIR / "leftImg8bit"
    CITYSCAPES_GTFINE  = BASE_DIR / "gtFine"
    GT_ZIP_PATH     = BASE_DIR / "BOSS_recordings.yolov8.zip"
    TEST_FRAMES_DIR = BASE_DIR / "recordings" / "20260514_092949" / "frames"

# CITYSCAPES_RAW: cartella virtuale che punta a leftImg8bit e gtFine
# Su Kaggle i due dataset sono separati; CITYSCAPES_RAW non è una cartella fisica
# — usa CITYSCAPES_LEFTIMG e CITYSCAPES_GTFINE direttamente nelle funzioni di conversione.

# Classi ostacoli per B.O.S.S.
BOSS_CLASSES = [
    "person",        # 0
    "car",           # 1
    "truck",         # 2
    "bus",           # 3
    "bicycle",       # 4
    "motorcycle",    # 5
    "traffic light", # 6
    "fire hydrant",  # 7
    "stop sign",     # 8
    "bench",         # 9
    "pole",          # 10
    "chair",         # 11
    "table",         # 12
    "sofa",          # 13
    "potted plant",  # 14
    "door",          # 15
    "stroller",      # 16
    "trash bin",     # 17
    "tree",          # 18
]

NUM_CLASSES = len(BOSS_CLASSES)  # 19

EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
LEARNING_RATE = 0.01
DEVICE = "0" if torch.cuda.is_available() else "cpu"

CONF_THRESHOLD_TRAIN = 0.25
CONF_THRESHOLD_TEST  = 0.50
IOU_THRESHOLD        = 0.45

print(f"Modello: {MODEL_PATH}")
print(f"Classi ({NUM_CLASSES}): {BOSS_CLASSES}")
print(f"Dataset train (Cityscapes): {DATASET_DIR}")
print(f"Frame test (recordings):    {TEST_FRAMES_DIR}")
if IS_KAGGLE:
    print(f"Cityscapes leftImg8bit:     {CITYSCAPES_LEFTIMG}")
    print(f"Cityscapes gtFine:          {CITYSCAPES_GTFINE}")
    print(f"Ground truth dir:           {GT_DIR}")
else:
    print(f"Ground truth ZIP:           {GT_ZIP_PATH}")
print(f"Conf threshold train: {CONF_THRESHOLD_TRAIN} | test: {CONF_THRESHOLD_TEST}")

Modello: yolov8n.pt
Classi (19): ['person', 'car', 'truck', 'bus', 'bicycle', 'motorcycle', 'traffic light', 'fire hydrant', 'stop sign', 'bench', 'pole', 'chair', 'table', 'sofa', 'potted plant', 'door', 'stroller', 'trash bin', 'tree']
Dataset train (Cityscapes): /kaggle/working/dataset_boss
Frame test (recordings):    /kaggle/input/datasets/lorenzoverdura/boss-recordings1/recordings/recordings/20260514_092949/frames
Cityscapes leftImg8bit:     /kaggle/input/datasets/chrisviviers/cityscapes-leftimg8bit-trainvaltest/leftImg8bit
Cityscapes gtFine:          /kaggle/input/datasets/lorenzoverdura/cityscapes-gtfine-trainvaltest/gtFine
Ground truth dir:           /kaggle/input/datasets/lorenzoverdura/boss-recordings1/BOSS_recordings.yolov8
Conf threshold train: 0.25 | test: 0.5


## Cella 4 — Verifica percorsi dataset

**Su Kaggle:** i dataset sono già estratti in `/kaggle/input/`. La cella verifica che tutti i percorsi esistano e conta i file disponibili. Nessuna estrazione o copia viene eseguita.

**In locale/Colab:** estrae gli ZIP da BASE_DIR:
- `leftImg8bit_trainvaltest.zip` → `leftImg8bit/`
- `gtFine_trainvaltest.zip` → `gtFine/`
- `recordings.zip` → `recordings/`

`BOSS_recordings.yolov8.zip` viene gestito separatamente nella Cella 16.

In [21]:
import zipfile

if IS_KAGGLE:
    # Verifica che tutti i percorsi di input esistano
    checks = [
        ("leftImg8bit", CITYSCAPES_LEFTIMG),
        ("gtFine",      CITYSCAPES_GTFINE),
        ("GT_DIR",      GT_DIR),
        ("frames test", TEST_FRAMES_DIR),
    ]
    for name, path in checks:
        if path.exists():
            print(f"OK  {name}: {path}")
        else:
            print(f"MANCANTE  {name}: {path}")

    n = len(list(TEST_FRAMES_DIR.glob("*.jpg"))) + len(list(TEST_FRAMES_DIR.glob("*.png"))) if TEST_FRAMES_DIR.exists() else 0
    print(f"\nFrame test trovati: {n}")
else:
    # Locale/Colab: estrae gli ZIP
    zip_specs = [
        ("leftImg8bit_trainvaltest.zip", CITYSCAPES_LEFTIMG, BASE_DIR),
        ("gtFine_trainvaltest.zip",      CITYSCAPES_GTFINE,  BASE_DIR),
        ("recordings.zip",               BASE_DIR / "recordings", BASE_DIR),
    ]
    for zip_name, expected_dir, extract_to in zip_specs:
        zip_path = BASE_DIR / zip_name
        if expected_dir.exists() and any(expected_dir.iterdir()):
            print(f"OK  {expected_dir.name}/ già presente")
            continue
        if not zip_path.exists():
            print(f"MANCANTE {zip_name}")
            continue
        print(f"Estrazione {zip_name} ...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_to)
        print(f"  -> {expected_dir}")

AttributeError: 'str' object has no attribute 'exists'

## Cella 5 — Conversione Cityscapes → formato YOLO
Converte le annotazioni poligonali di Cityscapes in bounding box YOLO normalizzate.
Filtra solo le classi definite in `BOSS_CLASSES`.

**Esegui dopo la Cella 4 (verifica/estrazione Cityscapes).**

In [ ]:
# Mappa dalle classi Cityscapes alle classi B.O.S.S.
CITYSCAPES_TO_BOSS = {
    "person":        "person",
    "rider":         "person",
    "car":           "car",
    "truck":         "truck",
    "bus":           "bus",
    "bicycle":       "bicycle",
    "motorcycle":    "motorcycle",
    "pole":          "pole",
    "traffic light": "traffic light",
    "traffic sign":  "stop sign",
}

CLASS_TO_ID = {cls: idx for idx, cls in enumerate(BOSS_CLASSES)}


def cityscapes_polygon_to_yolo_bbox(polygon, img_w, img_h):
    xs = [pt[0] for pt in polygon]
    ys = [pt[1] for pt in polygon]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = ((x_min + x_max) / 2) / img_w
    y_center = ((y_min + y_max) / 2) / img_h
    width    = (x_max - x_min) / img_w
    height   = (y_max - y_min) / img_h
    return x_center, y_center, width, height


def convert_cityscapes_split(img_root_base, ann_root_base, split, output_dir):
    """Converte uno split Cityscapes in dataset YOLO.
    img_root_base: percorso a leftImg8bit/ (senza lo split)
    ann_root_base: percorso a gtFine/ (senza lo split)
    """
    img_root  = img_root_base / split
    ann_root  = ann_root_base / split
    out_imgs  = output_dir / "images" / split
    out_lbls  = output_dir / "labels" / split
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_lbls.mkdir(parents=True, exist_ok=True)

    if not ann_root.exists():
        print(f"  {split}: annotazioni non trovate in {ann_root} - saltato")
        return

    json_files = list(ann_root.rglob("*_gtFine_polygons.json"))
    if not json_files:
        print(f"  {split}: nessun file gtFine_polygons.json - saltato")
        return

    converted, skipped = 0, 0
    for json_path in tqdm(json_files, desc=f"Conversione {split}"):
        with open(json_path) as f:
            data = json.load(f)
        img_w, img_h = data["imgWidth"], data["imgHeight"]

        lines = []
        for obj in data["objects"]:
            city_label = obj["label"]
            if city_label not in CITYSCAPES_TO_BOSS:
                continue
            class_id = CLASS_TO_ID[CITYSCAPES_TO_BOSS[city_label]]
            polygon  = obj["polygon"]
            if len(polygon) < 3:
                continue
            xc, yc, w, h = cityscapes_polygon_to_yolo_bbox(polygon, img_w, img_h)
            if w < 0.005 or h < 0.005:
                continue
            lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        if not lines:
            skipped += 1
            continue

        city = json_path.parent.name
        stem = json_path.name.replace("_gtFine_polygons.json", "")
        img_name = f"{stem}_leftImg8bit.png"
        img_src  = img_root / city / img_name
        if not img_src.exists():
            skipped += 1
            continue

        shutil.copy(img_src, out_imgs / img_name)
        (out_lbls / img_name.replace(".png", ".txt")).write_text("\n".join(lines))
        converted += 1

    print(f"  {split}: {converted} immagini convertite, {skipped} saltate")


if CITYSCAPES_LEFTIMG.exists() and CITYSCAPES_GTFINE.exists():
    for split in ["train", "val"]:
        convert_cityscapes_split(CITYSCAPES_LEFTIMG, CITYSCAPES_GTFINE, split, DATASET_DIR)
    print("Conversione completata.")
else:
    print(f"Cityscapes non trovato. leftImg8bit: {CITYSCAPES_LEFTIMG} | gtFine: {CITYSCAPES_GTFINE}")

## Cella 6 — Auto-annotazione classi mancanti con YOLOv8 pre-trained

Cityscapes copre solo 10 delle 19 classi BOSS. Le rimanenti classi presenti in COCO
(`fire hydrant`, `bench`, `chair`, `table`, `sofa`, `potted plant`) vengono auto-annotate
usando `yolov8n.pt` (già pre-trained su COCO) come modello di etichettatura.

Le predizioni vengono **aggiunte** ai file `.txt` YOLO esistenti (no sovrascrittura).
Le 4 classi rimanenti (`door`, `stroller`, `trash bin`, `tree`) restano senza training data:
verranno comunque incluse nel modello finale ma non saranno apprese (AP=0 su quelle classi).

**Tempo stimato su CPU i5-1035G1:** 30-50 min su ~3000 immagini.
**Salta questa cella** se userai Colab con GPU (li puoi rifare in <5 min) e se intendi
prima testare la pipeline su poche epoche.


In [ ]:
# Auto-annotazione delle classi BOSS coperte da COCO ma non da Cityscapes.
# Usa yolov8n.pt (gia' in cartella, pre-trained su COCO) per generare bbox aggiuntive.

# Mappa: COCO class ID -> nome BOSS (solo classi mancanti in Cityscapes)
# Indici COCO standard (80 classi)
COCO_TO_BOSS_NAME = {
    10: "fire hydrant",   # COCO 'fire hydrant'
    13: "bench",          # COCO 'bench'
    56: "chair",          # COCO 'chair'
    57: "sofa",           # COCO 'couch' -> mappato a sofa
    58: "potted plant",   # COCO 'potted plant'
    60: "table",          # COCO 'dining table' -> mappato a table
}

# Solo le classi COCO da considerare nelle predizioni
COCO_CLASS_FILTER = list(COCO_TO_BOSS_NAME.keys())

# Carica YOLOv8 pre-trained (gia' presente in BASE_DIR/yolov8n.pt)
auto_annot_model = YOLO(str(BASE_DIR / "yolov8n.pt"))

# Soglia conservativa: meglio meno annotazioni ma piu' affidabili
AUTO_ANNOT_CONF = 0.35


def auto_annotate_split(split):
    """Predice con yolov8n su tutte le immagini di uno split e APPENDE
    le bbox delle classi COCO->BOSS ai file .txt YOLO esistenti."""
    img_dir   = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split

    images = list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpg"))
    if not images:
        print(f"  {split}: nessuna immagine trovata in {img_dir}")
        return

    added_total = 0
    # stream=True per non saturare la RAM su grandi dataset
    results_iter = auto_annot_model.predict(
        source   = [str(p) for p in images],
        conf     = AUTO_ANNOT_CONF,
        iou      = 0.5,
        imgsz    = 640,
        device   = DEVICE,
        classes  = COCO_CLASS_FILTER,   # restringe l'inferenza alle sole classi che ci interessano
        stream   = True,
        verbose  = False,
        save     = False,
    )

    for result in tqdm(results_iter, total=len(images), desc=f"Auto-annot {split}"):
        img_path = Path(result.path)
        label_path = label_dir / (img_path.stem + ".txt")

        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue

        new_lines = []
        # xywhn = bbox in formato YOLO normalizzato (cx, cy, w, h)
        for i in range(len(boxes)):
            coco_id = int(boxes.cls[i].item())
            boss_name = COCO_TO_BOSS_NAME.get(coco_id)
            if boss_name is None:
                continue
            boss_id = CLASS_TO_ID.get(boss_name)
            if boss_id is None:
                continue
            xc, yc, w, h = boxes.xywhn[i].cpu().numpy().tolist()
            new_lines.append(f"{boss_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        if not new_lines:
            continue

        # APPEND alle annotazioni Cityscapes esistenti (non sovrascrive)
        with open(label_path, "a") as f:
            f.write("\n" + "\n".join(new_lines))
        added_total += len(new_lines)

    print(f"  {split}: {added_total} bbox aggiunte")


if not (BASE_DIR / "yolov8n.pt").exists():
    print("yolov8n.pt non trovato in BASE_DIR. Saltato.")
else:
    print(f"Auto-annotazione classi: {list(COCO_TO_BOSS_NAME.values())}")
    print(f"Soglia confidenza: {AUTO_ANNOT_CONF}\n")
    for split in ["train", "val"]:
        auto_annotate_split(split)
    print("\nAuto-annotazione completata.")


## Cella 7 — Creazione file data.yaml per YOLOv8

In [ ]:
# Genera il file data.yaml richiesto da YOLOv8 per il training.
# Contiene i percorsi dei dataset e la definizione delle classi.

import yaml

data_yaml = {
    "path": str(DATASET_DIR),                              # percorso radice del dataset
    "train": "images/train",                              # cartella immagini training
    "val":   "images/val",                                # cartella immagini validazione
    "test":  "images/test",                               # cartella immagini test
    "nc":    NUM_CLASSES,                                 # numero di classi
    "names": BOSS_CLASSES,                                # lista nomi classi
}

yaml_path = DATASET_DIR / "data.yaml"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f"data.yaml scritto in: {yaml_path}")
print(yaml.dump(data_yaml, default_flow_style=False))

## Cella 8 — Statistiche dataset pre-training

In [ ]:
# Analizza la distribuzione delle classi nel dataset prima del training.
# Importante per individuare class imbalance che potrebbe degradare le performance.

def count_class_distribution(label_dir):
    """
    Conta le occorrenze di ogni classe in tutti i file .txt YOLO di una cartella.
    Restituisce un dict {class_id: count}.
    """
    counts = {i: 0 for i in range(NUM_CLASSES)}
    label_files = list(Path(label_dir).rglob("*.txt"))

    # Per ogni file di annotazione: incrementa il contatore per ogni riga (= un oggetto)
    for lf in label_files:
        for line in lf.read_text().strip().split("\n"):
            if line.strip():               # salta righe vuote
                class_id = int(line.split()[0])   # primo valore = ID classe
                if class_id in counts:
                    counts[class_id] += 1
    return counts


train_counts = count_class_distribution(DATASET_DIR / "labels" / "train")
val_counts   = count_class_distribution(DATASET_DIR / "labels" / "val")

# Costruisce DataFrame per visualizzazione tabellare
df_dist = pd.DataFrame({
    "Classe":   BOSS_CLASSES,
    "Train":    [train_counts[i] for i in range(NUM_CLASSES)],
    "Val":      [val_counts[i]   for i in range(NUM_CLASSES)],
})
df_dist["Totale"] = df_dist["Train"] + df_dist["Val"]

print(df_dist.to_string(index=False))

# Grafico a barre della distribuzione delle classi nel training set
fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab20.colors[:NUM_CLASSES]   # colore diverso per ogni classe
bars = ax.bar(BOSS_CLASSES, df_dist["Train"], color=colors)
ax.set_title("Distribuzione classi — Training set", fontsize=14)
ax.set_xlabel("Classe")
ax.set_ylabel("Numero istanze")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(BASE_DIR / "plot_class_distribution.png", dpi=150)
plt.show()

## Cella 9 — Training YOLOv8
Il training salva automaticamente i pesi migliori in `runs/detect/boss_yolo/weights/best.pt`.

In [ ]:
# Fine-tuning del modello YOLOv8 sul dataset B.O.S.S.
# Tutti i parametri sono configurabili nella cella 3.

model = YOLO(MODEL_PATH)   # carica il modello di partenza (pre-trained su COCO)

results_train = model.train(
    data      = str(yaml_path),      # file data.yaml con percorsi e classi
    epochs    = EPOCHS,              # numero di epoche di training
    batch     = BATCH_SIZE,          # immagini per batch
    imgsz     = IMG_SIZE,            # risoluzione input (640x640)
    lr0       = LEARNING_RATE,       # learning rate iniziale
    device    = DEVICE,              # GPU/CPU
    name      = "boss_yolo",         # nome cartella di output in runs/detect/
    patience  = 15,                  # early stopping: ferma se val/mAP non migliora per 15 epoche
    save      = True,                # salva best.pt e last.pt
    plots     = True,                # genera grafici training automaticamente
    verbose   = True,
)

# Percorso del modello migliore salvato
BEST_MODEL_PATH = Path(results_train.save_dir) / "weights" / "best.pt"
print(f"Modello migliore salvato in: {BEST_MODEL_PATH}")

## Cella 10 — Validazione sul validation set
Calcola mAP, Precision, Recall sul validation set usando il modello migliore.

In [ ]:
# Se il training non è stato eseguito in questa sessione, carica il modello dal percorso salvato
# BEST_MODEL_PATH = Path("runs/detect/boss_yolo/weights/best.pt")  # decommenta e adatta

trained_model = YOLO(str(BEST_MODEL_PATH))

# Validazione interna su Cityscapes (val split) — soglia bassa per metriche di apprendimento
val_results = trained_model.val(
    data   = str(yaml_path),
    imgsz  = IMG_SIZE,
    conf   = CONF_THRESHOLD_TRAIN,   # soglia bassa per fase di apprendimento/val interna
    iou    = IOU_THRESHOLD,
    device = DEVICE,
    plots  = True,     # genera confusion matrix e curve PR automaticamente
)

# Estrae le metriche aggregate
map50    = val_results.box.map50       # mAP @ IoU=0.5
map5095  = val_results.box.map         # mAP @ IoU=0.5:0.95
precision = val_results.box.mp         # mean Precision
recall    = val_results.box.mr         # mean Recall
f1_score  = 2 * (precision * recall) / (precision + recall + 1e-9)  # F1 calcolato da P e R

print("\n=== Metriche Validazione interna (Cityscapes val split) ===")
print(f"mAP@0.5:       {map50:.4f}")
print(f"mAP@0.5:0.95:  {map5095:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1 Score:      {f1_score:.4f}")

## Cella 11 — Metriche per classe e tabella riepilogativa

In [ ]:
# Costruisce una tabella con le metriche per ogni singola classe.
# Utile per identificare quali classi hanno performance peggiori.

# Estrae le metriche per classe dall'oggetto risultati di validazione
per_class_ap50   = val_results.box.ap50         # AP@0.5 per classe (array)
per_class_ap5095 = val_results.box.ap            # AP@0.5:0.95 per classe (array)
per_class_p      = val_results.box.p             # Precision per classe
per_class_r      = val_results.box.r             # Recall per classe

# Calcola F1 per ogni classe come media armonica di P e R
per_class_f1 = 2 * (per_class_p * per_class_r) / (per_class_p + per_class_r + 1e-9)

# Costruisce il DataFrame delle metriche per classe
df_metrics = pd.DataFrame({
    "Classe":       BOSS_CLASSES[:len(per_class_ap50)],
    "AP@0.5":       per_class_ap50,
    "AP@0.5:0.95":  per_class_ap5095,
    "Precision":    per_class_p,
    "Recall":       per_class_r,
    "F1":           per_class_f1,
})

# Riga riassuntiva con i valori medi
summary_row = pd.DataFrame([{
    "Classe":      "MEDIA",
    "AP@0.5":      df_metrics["AP@0.5"].mean(),
    "AP@0.5:0.95": df_metrics["AP@0.5:0.95"].mean(),
    "Precision":   df_metrics["Precision"].mean(),
    "Recall":      df_metrics["Recall"].mean(),
    "F1":          df_metrics["F1"].mean(),
}])
df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)

# Formattazione numerica
float_cols = ["AP@0.5", "AP@0.5:0.95", "Precision", "Recall", "F1"]
df_display = df_metrics.copy()
df_display[float_cols] = df_display[float_cols].applymap(lambda x: f"{x:.4f}")

print("\n=== Metriche per Classe ===")
print(df_display.to_string(index=False))

# Salva come CSV per riferimento futuro
df_metrics.to_csv(BASE_DIR / "metrics_per_class.csv", index=False)
print("\nSalvato: metrics_per_class.csv")

## Cella 12 — Grafico AP per classe

In [ ]:
# Visualizza AP@0.5 e AP@0.5:0.95 per ogni classe in un grafico a barre affiancate.
# Permette di confrontare immediatamente le performance per classe.

df_plot = df_metrics[df_metrics["Classe"] != "MEDIA"].copy()   # esclude la riga media

x = np.arange(len(df_plot))   # posizioni sull'asse x
width = 0.35                   # larghezza di ogni barra

fig, ax = plt.subplots(figsize=(16, 6))
bars1 = ax.bar(x - width/2, df_plot["AP@0.5"],      width, label="AP@0.5",      color="steelblue")
bars2 = ax.bar(x + width/2, df_plot["AP@0.5:0.95"], width, label="AP@0.5:0.95", color="coral")

# Aggiunge etichette numeriche sopra ogni barra
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_title("Average Precision per Classe", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(df_plot["Classe"], rotation=45, ha="right")
ax.set_ylabel("AP")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / "plot_ap_per_class.png", dpi=150)
plt.show()

## Cella 13 — Curva Precision-Recall

In [ ]:
# Traccia la curva Precision-Recall media (mAP) e per singola classe.
# L'area sotto la curva PR = AP. Curve verso l'alto-destra indicano modello migliore.

# YOLOv8 salva la curva PR in runs/detect/boss_yolo/PR_curve.png
# Questa cella la rielabora in forma personalizzata

pr_curve_path = Path(results_train.save_dir) / "PR_curve.png"

if pr_curve_path.exists():
    # Mostra la curva PR generata automaticamente da YOLOv8
    img_pr = Image.open(pr_curve_path)
    plt.figure(figsize=(10, 7))
    plt.imshow(img_pr)
    plt.axis("off")
    plt.title("Curva Precision-Recall (generata da YOLOv8)")
    plt.tight_layout()
    plt.show()
else:
    # Se la curva non è disponibile, traccia i punti P/R per classe
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, cls in enumerate(BOSS_CLASSES[:len(per_class_p)]):
        ax.scatter(per_class_r[i], per_class_p[i], label=cls, s=80)

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision vs Recall per Classe")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(BASE_DIR / "plot_pr_scatter.png", dpi=150)
    plt.show()

## Cella 14 — Confusion Matrix

In [ ]:
# Visualizza la confusion matrix normalizzata per ogni classe.
# Le righe rappresentano i label reali (ground truth),
# le colonne le predizioni del modello.
# La diagonale principale = classificazioni corrette.

cm_path = Path(results_train.save_dir) / "confusion_matrix_normalized.png"

if cm_path.exists():
    # Mostra la confusion matrix salvata da YOLOv8
    img_cm = Image.open(cm_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img_cm)
    plt.axis("off")
    plt.title("Confusion Matrix Normalizzata (generata da YOLOv8)")
    plt.tight_layout()
    plt.show()
else:
    # Fallback: costruisce la confusion matrix manualmente dai risultati di validazione
    # Usa le predizioni raw memorizzate nell'oggetto val_results
    print("Confusion matrix non trovata nel percorso standard.")
    print(f"Cerca in: {Path(results_train.save_dir)}")

## Cella 15 — Curve di training (loss, mAP per epoca)

In [ ]:
# Legge il file results.csv generato da YOLOv8 durante il training
# e traccia le curve di loss e mAP per epoca.
# Utile per diagnosticare overfitting, underfitting o learning rate inadeguato.

results_csv = Path(results_train.save_dir) / "results.csv"

if not results_csv.exists():
    print(f"File results.csv non trovato in {results_train.save_dir}")
else:
    df_train = pd.read_csv(results_csv)
    # Pulisce i nomi delle colonne (YOLOv8 può aggiungere spazi)
    df_train.columns = df_train.columns.str.strip()

    print("Colonne disponibili:", list(df_train.columns))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Curve di Training — B.O.S.S. YOLOv8", fontsize=15)

    # Definisce i pannelli da tracciare: (colonna_csv, titolo_pannello, colore)
    panels = [
        ("train/box_loss",  "Train Box Loss",       "steelblue"),
        ("train/cls_loss",  "Train Classification Loss", "coral"),
        ("train/dfl_loss",  "Train DFL Loss",        "green"),
        ("val/box_loss",    "Val Box Loss",           "steelblue"),
        ("metrics/mAP50",   "mAP@0.5",               "purple"),
        ("metrics/mAP50-95","mAP@0.5:0.95",          "orange"),
    ]

    # Per ogni pannello: traccia la metrica per epoca
    for ax, (col, title, color) in zip(axes.flat, panels):
        if col in df_train.columns:
            ax.plot(df_train["epoch"], df_train[col], color=color, linewidth=1.5)
            ax.set_title(title, fontsize=11)
            ax.set_xlabel("Epoca")
            ax.grid(True, alpha=0.3)   # griglia leggera per facilità di lettura
        else:
            ax.text(0.5, 0.5, f"Colonna '{col}' non trovata",
                    ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title)

    plt.tight_layout()
    plt.savefig(BASE_DIR / "plot_training_curves.png", dpi=150)
    plt.show()

## Cella 16 — Preparazione Ground Truth

I frame in `recordings/.../frames` sono l'**input dell'inferenza** (immagini grezze, non annotate).
La cartella/ZIP Roboflow contiene gli **stessi frame annotati** e fa da **ground truth**.

**Su Kaggle:** `GT_DIR` = `/kaggle/input/boss_recordings1/BOSS_recordings/` già estratta — nessuna estrazione.
**In locale/Colab:** `GT_ZIP_PATH` = `BOSS_recordings.yolov8.zip` — viene estratto in `GT_EXTRACTED/`.

In entrambi i casi questa cella:
1. legge `data.yaml` di Roboflow per ottenere l'ordine classi originale
2. costruisce la mappa `RF_id → BOSS_id` per allineare gli ID classi
3. copia immagini/label remappate in `ground_truth_boss/test/`
4. scrive `data_boss.yaml` che YOLOv8 userà per `val(split="test")`

In [ ]:
import zipfile

# 1) Determina la cartella sorgente della ground truth
if IS_KAGGLE:
    # Su Kaggle BOSS_recordings/ è già estratta come dataset di input
    gt_src_root = GT_DIR
    print(f"Ground truth da input Kaggle: {gt_src_root}")
else:
    # In locale/Colab: estrai lo ZIP in GT_EXTRACTED
    if not GT_ZIP_PATH.exists():
        raise FileNotFoundError(f"ZIP ground truth non trovato: {GT_ZIP_PATH}")
    GT_EXTRACTED.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(GT_ZIP_PATH, "r") as z:
        z.extractall(GT_EXTRACTED)
    print(f"ZIP estratto in: {GT_EXTRACTED}")
    gt_src_root = GT_EXTRACTED

# 2) Leggi il data.yaml originale di Roboflow
gt_yaml_in_path = gt_src_root / "data.yaml"
with open(gt_yaml_in_path) as f:
    gt_yaml_in = yaml.safe_load(f)
RF_CLASSES = gt_yaml_in["names"]
print(f"Classi Roboflow ({len(RF_CLASSES)}): {RF_CLASSES}")

# 3) Mappa RF_id → BOSS_id (case-insensitive)
RF_TO_BOSS_ID = {}
for rf_id, rf_name in enumerate(RF_CLASSES):
    rf_name_lower = rf_name.lower().strip()
    for boss_id, boss_name in enumerate(BOSS_CLASSES):
        if rf_name_lower == boss_name.lower():
            RF_TO_BOSS_ID[rf_id] = boss_id
            break

unmapped = [RF_CLASSES[i] for i in range(len(RF_CLASSES)) if i not in RF_TO_BOSS_ID]
print(f"Mappa RF→BOSS: {RF_TO_BOSS_ID}")
print(f"Classi Roboflow scartate (non in BOSS_CLASSES): {unmapped}")


def remap_labels(src_label_dir, dst_label_dir, id_map):
    src = Path(src_label_dir)
    dst = Path(dst_label_dir)
    dst.mkdir(parents=True, exist_ok=True)
    remapped, dropped = 0, 0
    for lf in src.glob("*.txt"):
        out_lines = []
        for line in lf.read_text().strip().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            rf_id = int(parts[0])
            if rf_id not in id_map:
                dropped += 1
                continue
            parts[0] = str(id_map[rf_id])
            out_lines.append(" ".join(parts))
            remapped += 1
        (dst / lf.name).write_text("\n".join(out_lines))
    print(f"  Annotazioni rimappate: {remapped} | Righe scartate: {dropped}")


# 4) Copia immagini e label remappate in GT_EXTRACTED/test/ (working dir scrivibile)
GT_SRC_IMGS    = gt_src_root / "train" / "images"
GT_SRC_LABELS  = gt_src_root / "train" / "labels"
GT_TEST_IMGS   = GT_EXTRACTED / "test" / "images"
GT_TEST_LABELS = GT_EXTRACTED / "test" / "labels"

GT_EXTRACTED.mkdir(parents=True, exist_ok=True)
GT_TEST_IMGS.mkdir(parents=True, exist_ok=True)
GT_TEST_LABELS.mkdir(parents=True, exist_ok=True)

img_count = 0
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for img in GT_SRC_IMGS.glob(ext):
        shutil.copy(img, GT_TEST_IMGS / img.name)
        img_count += 1
print(f"Immagini copiate in test/images: {img_count}")

remap_labels(GT_SRC_LABELS, GT_TEST_LABELS, RF_TO_BOSS_ID)

# 5) Scrivi data_boss.yaml nella working dir
gt_data_yaml = {
    "path":  str(GT_EXTRACTED),
    "train": "",
    "val":   "",
    "test":  "test/images",
    "nc":    NUM_CLASSES,
    "names": BOSS_CLASSES,
}
gt_yaml_out = GT_EXTRACTED / "data_boss.yaml"
with open(gt_yaml_out, "w") as f:
    yaml.dump(gt_data_yaml, f, default_flow_style=False, allow_unicode=True)
print(f"\nGround truth data.yaml scritto in: {gt_yaml_out}")

## Cella 17 — Inferenza sui frame `recordings/` + Valutazione contro Ground Truth

Due fasi distinte:

**A) Inferenza visuale** sui frame in `TEST_FRAMES_DIR` (`recordings/.../frames`).
   Produce immagini con bounding box disegnate, salvate in `runs/detect/boss_test_frames/`.
   Soglia: `CONF_THRESHOLD_TEST` (alta = pochi falsi positivi).

**B) Metriche quantitative** via `trained_model.val(data=data_boss.yaml, split="test")`.
   YOLOv8 confronta le predizioni del modello con le label remappate in `ground_truth_boss/test/labels/`.
   Restituisce mAP@0.5, mAP@0.5:0.95, Precision, Recall per classe e aggregati, oltre alla confusion matrix.

In [ ]:
# ------------------------------------------------------------
# FASE A — Inferenza sui frame "recordings/" con visualizzazione
# ------------------------------------------------------------
# Input:  TEST_FRAMES_DIR (frame grezzi, non annotati)
# Output: cartella runs/detect/boss_test_frames/ con immagini predette + .txt YOLO

# Conta i frame disponibili
test_frames = sorted(TEST_FRAMES_DIR.glob("*.jpg")) + sorted(TEST_FRAMES_DIR.glob("*.png"))
print(f"Frame trovati in {TEST_FRAMES_DIR}: {len(test_frames)}")
if len(test_frames) == 0:
    raise FileNotFoundError(f"Nessun frame trovato in {TEST_FRAMES_DIR}")

# Esegue l'inferenza:
# - stream=True: processa un frame alla volta (memoria contenuta)
# - save=True / save_txt=True: salva immagini annotate e file label
# - exist_ok=False: forza la creazione di un nome univoco se la cartella esiste già
infer_run_name = "boss_test_frames"
inference_results = trained_model.predict(
    source   = str(TEST_FRAMES_DIR),
    conf     = CONF_THRESHOLD_TEST,    # soglia alta per la fase di test
    iou      = IOU_THRESHOLD,
    imgsz    = IMG_SIZE,
    device   = DEVICE,
    save     = True,
    save_txt = True,
    name     = infer_run_name,
    exist_ok = False,
    stream   = True,
)

# Raccoglie le predizioni in un DataFrame e individua in modo robusto la cartella di output
# (usa result.save_dir restituito dal predict, non un glob su runs/detect/).
all_predictions = []
PREDICT_SAVE_DIR = None

# Per ogni frame: estrae le bounding box predette e i metadati
for result in inference_results:
    if PREDICT_SAVE_DIR is None:
        PREDICT_SAVE_DIR = Path(result.save_dir)   # cartella esatta della run corrente

    frame_name = Path(result.path).name
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        continue

    # Per ogni bbox del frame: estrae class_id, confidenza e coordinate pixel
    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i].item())
        conf   = float(boxes.conf[i].item())
        xyxy   = boxes.xyxy[i].cpu().numpy()
        all_predictions.append({
            "frame":      frame_name,
            "class_id":   cls_id,
            "class_name": BOSS_CLASSES[cls_id] if cls_id < NUM_CLASSES else "unknown",
            "confidence": conf,
            "x1": xyxy[0], "y1": xyxy[1], "x2": xyxy[2], "y2": xyxy[3],
        })

df_preds = pd.DataFrame(all_predictions)
print(f"Cartella inferenza: {PREDICT_SAVE_DIR}")
print(f"Totale predizioni:  {len(df_preds)}")
if not df_preds.empty:
    print(df_preds.head(10).to_string(index=False))
df_preds.to_csv(BASE_DIR / "predictions_test_frames.csv", index=False)


# ------------------------------------------------------------
# FASE B — Valutazione quantitativa contro la Ground Truth
# ------------------------------------------------------------
# trained_model.val() confronta le predizioni del modello con le label in
# ground_truth_boss/test/labels/ (split="test" del data.yaml ground truth).
# Restituisce mAP/Precision/Recall per classe + confusion matrix.
gt_eval_run = "boss_gt_eval"
gt_val_results = trained_model.val(
    data     = str(gt_yaml_out),
    split    = "test",
    imgsz    = IMG_SIZE,
    conf     = CONF_THRESHOLD_TEST,    # stessa soglia alta per coerenza con la fase A
    iou      = IOU_THRESHOLD,
    device   = DEVICE,
    plots    = True,                    # genera PR curve e confusion matrix
    name     = gt_eval_run,
    exist_ok = False,
)

# Estrae metriche aggregate (sostituiscono quelle interne di Cityscapes per il riepilogo finale)
map50     = gt_val_results.box.map50
map5095   = gt_val_results.box.map
precision = gt_val_results.box.mp
recall    = gt_val_results.box.mr
f1_score  = 2 * (precision * recall) / (precision + recall + 1e-9)

# Salva il save_dir di val per i grafici della cella successiva (uso robusto, non glob)
GT_EVAL_SAVE_DIR = Path(gt_val_results.save_dir)

print("\n=== Metriche contro Ground Truth (recordings vs BOSS_recordings.yolov8) ===")
print(f"mAP@0.5:       {map50:.4f}")
print(f"mAP@0.5:0.95:  {map5095:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1 Score:      {f1_score:.4f}")
print(f"Plot/CM in:    {GT_EVAL_SAVE_DIR}")

## Cella 18 — Metriche per classe contro Ground Truth + grafici

Costruisce il DataFrame delle metriche per classe basato sui risultati di `val()` della cella 17
(non più sul validation set di Cityscapes). Visualizza AP per classe, P/R scatter e la confusion matrix
generata da YOLOv8 nella cartella `GT_EVAL_SAVE_DIR`.

In [ ]:
# Estrazione metriche per classe dall'oggetto val() contro Ground Truth
# Nota: alcune classi BOSS potrebbero non avere ground truth nello ZIP → AP/P/R = 0 per quelle.
gt_per_class_ap50   = gt_val_results.box.ap50    # AP@0.5 per classe (array di dimensione = numero classi con GT)
gt_per_class_ap5095 = gt_val_results.box.ap
gt_per_class_p      = gt_val_results.box.p
gt_per_class_r      = gt_val_results.box.r
gt_per_class_f1     = 2 * (gt_per_class_p * gt_per_class_r) / (gt_per_class_p + gt_per_class_r + 1e-9)

# YOLOv8 restituisce gli indici delle classi presenti nel test set (ap_class_index)
# per allinearli ai nomi BOSS corretti
gt_class_indices = gt_val_results.box.ap_class_index.astype(int)
gt_class_names   = [BOSS_CLASSES[i] for i in gt_class_indices]

# Costruisce il DataFrame delle metriche per classe sulla ground truth
df_metrics_gt = pd.DataFrame({
    "Classe":      gt_class_names,
    "AP@0.5":      gt_per_class_ap50,
    "AP@0.5:0.95": gt_per_class_ap5095,
    "Precision":   gt_per_class_p,
    "Recall":      gt_per_class_r,
    "F1":          gt_per_class_f1,
})

# Riga MEDIA sulle classi presenti nel test
summary_row_gt = pd.DataFrame([{
    "Classe":      "MEDIA",
    "AP@0.5":      df_metrics_gt["AP@0.5"].mean(),
    "AP@0.5:0.95": df_metrics_gt["AP@0.5:0.95"].mean(),
    "Precision":   df_metrics_gt["Precision"].mean(),
    "Recall":      df_metrics_gt["Recall"].mean(),
    "F1":          df_metrics_gt["F1"].mean(),
}])
df_metrics_gt = pd.concat([df_metrics_gt, summary_row_gt], ignore_index=True)

# Stampa tabellare formattata
float_cols = ["AP@0.5", "AP@0.5:0.95", "Precision", "Recall", "F1"]
df_display_gt = df_metrics_gt.copy()
df_display_gt[float_cols] = df_display_gt[float_cols].applymap(lambda x: f"{x:.4f}")
print("=== Metriche per Classe contro Ground Truth ===")
print(df_display_gt.to_string(index=False))
df_metrics_gt.to_csv(BASE_DIR / "metrics_per_class_gt.csv", index=False)

# Alias per le celle di riepilogo che usano df_metrics (compatibilità con le celle finali)
df_metrics = df_metrics_gt
per_class_ap50   = gt_per_class_ap50
per_class_ap5095 = gt_per_class_ap5095
per_class_p      = gt_per_class_p
per_class_r      = gt_per_class_r
per_class_f1     = gt_per_class_f1

# ----- Grafico 1: AP@0.5 e AP@0.5:0.95 per classe -----
df_plot_gt = df_metrics_gt[df_metrics_gt["Classe"] != "MEDIA"].copy()
x = np.arange(len(df_plot_gt))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, df_plot_gt["AP@0.5"],      width, label="AP@0.5",      color="steelblue")
bars2 = ax.bar(x + width/2, df_plot_gt["AP@0.5:0.95"], width, label="AP@0.5:0.95", color="coral")
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}",
            ha="center", va="bottom", fontsize=8)
ax.set_title("AP per classe — Ground Truth", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(df_plot_gt["Classe"], rotation=45, ha="right")
ax.set_ylabel("AP")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / "plot_ap_per_class_gt.png", dpi=150)
plt.show()

# ----- Grafico 2: scatter Precision vs Recall per classe -----
fig, ax = plt.subplots(figsize=(10, 7))
for i, row in df_plot_gt.iterrows():
    ax.scatter(row["Recall"], row["Precision"], s=100, zorder=5)
    ax.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision vs Recall per classe — Ground Truth")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / "plot_pr_scatter_gt.png", dpi=150)
plt.show()

# ----- Grafico 3: confusion matrix generata da YOLOv8 in GT_EVAL_SAVE_DIR -----
cm_gt_path = GT_EVAL_SAVE_DIR / "confusion_matrix_normalized.png"
if cm_gt_path.exists():
    img_cm = Image.open(cm_gt_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img_cm)
    plt.axis("off")
    plt.title("Confusion Matrix Normalizzata — Ground Truth")
    plt.tight_layout()
    plt.show()
else:
    print(f"Confusion matrix non trovata in {cm_gt_path}")

## Cella 19 — Campione frame annotati + Dashboard riepilogo finale

Mostra una griglia di frame con bbox predette (presi da `PREDICT_SAVE_DIR` restituito dal predict)
e un dashboard 6-pannelli che combina metriche aggregate, AP per classe, F1 per classe, scatter P/R,
distribuzione classi rilevate e configurazione.

In [ ]:
# ============================================================
# Parte 1: griglia campione frame annotati dalla cartella di inferenza
# ============================================================
import random

N_SAMPLES = 9   # numero di frame da visualizzare (idealmente quadrato perfetto)

# Usa PREDICT_SAVE_DIR (restituito da result.save_dir) invece di un glob fragile
if PREDICT_SAVE_DIR is None or not PREDICT_SAVE_DIR.exists():
    print(f"Cartella inferenza non disponibile: {PREDICT_SAVE_DIR}")
    annotated_imgs = []
else:
    annotated_imgs = list(PREDICT_SAVE_DIR.glob("*.jpg")) + list(PREDICT_SAVE_DIR.glob("*.png"))

if len(annotated_imgs) == 0:
    print("Nessun frame annotato disponibile per la griglia campione.")
else:
    sample = random.sample(annotated_imgs, min(N_SAMPLES, len(annotated_imgs)))
    grid_size = int(np.ceil(np.sqrt(len(sample))))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(5 * grid_size, 4 * grid_size))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    # Per ogni frame del campione: carica con OpenCV, converte BGR→RGB e visualizza
    for ax, img_path in zip(axes_flat, sample):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")
    # Nasconde gli assi rimasti vuoti se N_SAMPLES non riempie la griglia
    for ax in list(axes_flat)[len(sample):]:
        ax.axis("off")

    plt.suptitle("Campione frame recordings con predizioni B.O.S.S.", fontsize=14)
    plt.tight_layout()
    plt.savefig(BASE_DIR / "plot_sample_predictions.png", dpi=150)
    plt.show()


# ============================================================
# Parte 2: dashboard riepilogo 6-pannelli
# ============================================================
fig = plt.figure(figsize=(16, 10))
fig.suptitle("B.O.S.S. — Riepilogo Metriche (Ground Truth)", fontsize=16, fontweight="bold")

# Pannello 1 (top-left): metriche aggregate come barre orizzontali
ax1 = fig.add_subplot(2, 3, 1)
metrics_summary = {
    "mAP@0.5":      map50,
    "mAP@0.5:0.95": map5095,
    "Precision":    precision,
    "Recall":       recall,
    "F1 Score":     f1_score,
}
colors_gauge = ["#2196F3", "#1976D2", "#4CAF50", "#FF9800", "#9C27B0"]
bars_h = ax1.barh(list(metrics_summary.keys()), list(metrics_summary.values()),
                  color=colors_gauge)
for bar, val in zip(bars_h, metrics_summary.values()):
    ax1.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=10, fontweight="bold")
ax1.set_xlim(0, 1.1)
ax1.set_title("Metriche Aggregate vs GT", fontsize=11)
ax1.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="soglia 0.5")
ax1.legend(fontsize=8)

# Pannello 2 (top-center): AP@0.5 per classe
ax2 = fig.add_subplot(2, 3, 2)
df_plot = df_metrics[df_metrics["Classe"] != "MEDIA"]
ax2.barh(df_plot["Classe"], df_plot["AP@0.5"], color="steelblue")
ax2.set_title("AP@0.5 per Classe", fontsize=11)
ax2.set_xlim(0, 1)
ax2.axvline(map50, color="red", linestyle="--", alpha=0.7, label=f"media={map50:.2f}")
ax2.legend(fontsize=8)

# Pannello 3 (top-right): F1 per classe
ax3 = fig.add_subplot(2, 3, 3)
ax3.barh(df_plot["Classe"], df_plot["F1"], color="coral")
ax3.set_title("F1 Score per Classe", fontsize=11)
ax3.set_xlim(0, 1)
ax3.axvline(f1_score, color="red", linestyle="--", alpha=0.7, label=f"media={f1_score:.2f}")
ax3.legend(fontsize=8)

# Pannello 4 (bottom-left): scatter Precision vs Recall per classe
ax4 = fig.add_subplot(2, 3, 4)
scatter_colors = plt.cm.tab20.colors[:len(df_plot)]
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax4.scatter(row["Recall"], row["Precision"],
                color=scatter_colors[i % len(scatter_colors)],
                s=100, zorder=5)
    ax4.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                 textcoords="offset points", xytext=(5, 3), fontsize=7)
ax4.set_xlabel("Recall")
ax4.set_ylabel("Precision")
ax4.set_title("Precision vs Recall per Classe", fontsize=11)
ax4.set_xlim(0, 1.05)
ax4.set_ylim(0, 1.05)
ax4.grid(True, alpha=0.3)

# Pannello 5 (bottom-center): distribuzione classi rilevate sui frame recordings
ax5 = fig.add_subplot(2, 3, 5)
if not df_preds.empty:
    class_counts_test = df_preds["class_name"].value_counts()
    ax5.pie(class_counts_test.values,
            labels=class_counts_test.index,
            autopct="%1.1f%%",
            startangle=140,
            textprops={"fontsize": 8})
    ax5.set_title("Distribuzione classi — Frame recordings", fontsize=11)
else:
    ax5.text(0.5, 0.5, "Nessuna predizione", ha="center", va="center")
    ax5.axis("off")

# Pannello 6 (bottom-right): info configurazione
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis("off")
info_text = (
    f"Modello:           {MODEL_PATH}\n"
    f"Classi:            {NUM_CLASSES}\n"
    f"Epoche:            {EPOCHS}\n"
    f"Batch size:        {BATCH_SIZE}\n"
    f"Immagine:          {IMG_SIZE}x{IMG_SIZE}\n"
    f"Conf train:        {CONF_THRESHOLD_TRAIN}\n"
    f"Conf test:         {CONF_THRESHOLD_TEST}\n"
    f"IoU threshold:     {IOU_THRESHOLD}\n\n"
    f"Frame recordings:  {len(test_frames)}\n"
    f"Pred. totali:      {len(df_preds)}\n"
    f"GT img test:       {len(list(GT_TEST_IMGS.glob('*.jpg'))) + len(list(GT_TEST_IMGS.glob('*.png')))}\n"
)
ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
ax6.set_title("Configurazione", fontsize=11)

plt.tight_layout()
plt.savefig(BASE_DIR / "plot_metriche_riepilogo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvato: plot_metriche_riepilogo.png")